# Diagnose Fixed-Policy Test Failures
Replays only the 13 cases with original IoU below 0.1, using the installed release and unchanged policy. Records rejected detector/SAM outputs to distinguish filtering from missing masks. No training, reference changes or threshold selection.

Outputs are saved separately in Drive and downloaded as a diagnostic ZIP. Original test results remain unchanged.

If the release is missing, restores the exact verified ZIP from GitHub. You can enter custom release/evaluation paths. Use the Drive account holding the original test evaluation.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, json, hashlib, importlib.util, csv, zipfile
import pandas as pd
import torch

drive.mount('/content/drive')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.3.221'],check=True)
from ultralytics import YOLO, SAM
import urllib.request, shutil
RELEASE=Path(input('Release folder (Enter for default): ').strip() or '/content/drive/MyDrive/CameraTrap_Phase2/releases/Segmentation_M4_Guided')
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
if not (RELEASE/'refinement_policy.json').is_file():
    print('Release missing at:', RELEASE)
    print('Restoring the exact verified release from GitHub; no training.')
    archive=RELEASE.parent/'Segmentation_M4_Guided.zip'
    expected='4026e9d8335a9d213a875d7045849e76bba5b180c02f7a2914c9dab247fc1133'
    RELEASE.parent.mkdir(parents=True,exist_ok=True)
    if archive.exists():
        assert sha(archive)==expected, 'Existing archive differs; preserve it before retrying.'
    else:
        temporary=archive.with_suffix('.download')
        urllib.request.urlretrieve('https://raw.githubusercontent.com/salehdev3/Capstone_5/phase-2/releases/segmentation_m4_guided/Segmentation_M4_Guided.zip',temporary)
        assert sha(temporary)==expected, 'Archive checksum mismatch'
        temporary.replace(archive)
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None
        checks=json.loads(z.read('artifact_checksums.json'))
        for name,checksum in checks.items():assert hashlib.sha256(z.read(name)).hexdigest()==checksum,name
        for entry in z.infolist():
            target=(RELEASE/entry.filename).resolve()
            assert target.is_relative_to(RELEASE.resolve()), 'Unsafe archive path'
            if target.exists() and not entry.is_dir():
                assert target.is_file() and sha(target)==hashlib.sha256(z.read(entry.filename)).hexdigest(), f'Existing file differs: {target}'
        RELEASE.mkdir(parents=True,exist_ok=True)
        z.extractall(RELEASE)
    for name,checksum in checks.items():assert sha(RELEASE/name)==checksum,name
print('Using release:', RELEASE)
policy=json.loads((RELEASE/'refinement_policy.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
policy_hash=sha(RELEASE/'refinement_policy.json')
candidates=sorted(RELEASE.glob('evaluations/*/metrics.json'))
default_evaluation=str(candidates[-1].parent) if candidates else ''
if not candidates:
    print('Original evaluation is not in this release folder. Use the same Drive account as the test run, or enter its full evaluation folder path below.')
evaluation_input=input('Original evaluation folder (Enter for latest if found): ').strip() or default_evaluation
assert evaluation_input, 'Original test evaluation is required. Reconnect the Drive account used for the test; do not rerun the test just to resolve this path.'
EVAL=Path(evaluation_input)
assert (EVAL/'predictions.json').is_file() and (EVAL/'per_image.csv').is_file(), f'Original evaluation files missing: {EVAL}'
original=json.loads((EVAL/'predictions.json').read_text())
assert original['policy_sha256']==policy_hash, 'Policy differs from original evaluation'
frame=pd.read_csv(EVAL/'per_image.csv')
lookup={Path(r['image']).stem:r for r in original['results']}
assert len(frame)==40 and frame.id.is_unique and set(frame.id)==set(lookup)
assert sha(RELEASE/policy['model_file'])==policy['model_sha256']
assert sha(RELEASE/policy['detector_file'])==policy['detector_sha256']
sam_path=RELEASE/policy['sam_file']
if not sam_path.exists():
    for candidate in (Path.cwd()/policy['sam_file'],Path('/content')/policy['sam_file']):
        if candidate.is_file() and sha(candidate)==policy['sam_sha256']:
            shutil.copyfile(candidate,sam_path)
            break
if not sam_path.exists():
    print('Downloading SAM once for diagnostic inference.')
    temporary=sam_path.with_suffix('.download')
    urllib.request.urlretrieve(policy['sam_url'],temporary)
    assert sha(temporary)==policy['sam_sha256'], 'SAM checksum mismatch'
    temporary.replace(sam_path)
assert sha(sam_path)==policy['sam_sha256'], 'SAM checksum mismatch'
spec=importlib.util.spec_from_file_location('release_runtime',RELEASE/'predict_refined.py')
runtime=importlib.util.module_from_spec(spec);spec.loader.exec_module(runtime)
detector=YOLO(str(RELEASE/policy['detector_file']))
model=YOLO(str(RELEASE/policy['model_file']))
sam=SAM(str(RELEASE/policy['sam_file']))
DEVICE='0' if torch.cuda.is_available() else 'cpu'
OUT=RELEASE/'diagnostic_replays'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT.mkdir(parents=True,exist_ok=False)
trace={}
original_det_predict=detector.predict
original_sam_predict=sam.predict

def detector_spy(*args,**kwargs):
    outputs=original_det_predict(*args,**kwargs)
    trace['detector_confidences']=outputs[0].boxes.conf.cpu().numpy().tolist()[:policy['max_prompts']]
    trace['detector_boxes']=outputs[0].boxes.xyxy.cpu().numpy().tolist()[:policy['max_prompts']]
    return outputs

def sam_spy(*args,**kwargs):
    outputs=original_sam_predict(*args,**kwargs)
    result=outputs[0]
    trace['sam_outputs']=[dict(prompt_index=int(i),quality=float(q)) for i,q in zip(result.boxes.cls.cpu().numpy().tolist(),result.boxes.conf.cpu().numpy().tolist())] if result.masks is not None else []
    return outputs

detector.predict=detector_spy
sam.predict=sam_spy
summaries=[]
targets=frame.loc[frame.iou.lt(.1),'id'].tolist()
for number,key in enumerate(targets,1):
    trace.clear()
    path=Path(lookup[key]['image'])
    assert path.is_file(), path
    replay=runtime.predict_image(detector,model,sam,path,policy,DEVICE)
    confidences=trace.get('detector_confidences',[])
    outputs=trace.get('sam_outputs',[])
    prompt_details=[]
    for j,confidence in enumerate(confidences):
        qualities=[r['quality'] for r in outputs if r['prompt_index']==j]
        reasons=[]
        if confidence<policy['conf']:reasons.append('below_fixed_detector_confidence')
        if not qualities:reasons.append('no_SAM_mask_returned')
        elif max(qualities)<policy['sam_quality']:reasons.append('below_fixed_SAM_quality')
        prompt_details.append(dict(prompt_index=j,confidence=confidence,SAM_qualities=qualities,filter_reasons=reasons))
    detail=dict(id=key,policy_sha256=policy_hash,device=DEVICE,original_mask_count=lookup[key]['mask_count'],replayed_mask_count=replay['mask_count'],prompt_details=prompt_details,raw_trace=dict(trace),replayed_prediction=replay)
    (OUT/(key+'.json')).write_text(json.dumps(detail,indent=2))
    summaries.append(dict(id=key,original_iou=float(frame.loc[frame.id.eq(key),'iou'].iloc[0]),original_masks=lookup[key]['mask_count'],replayed_masks=replay['mask_count'],prompts=len(confidences),prompts_below_confidence=sum(c<policy['conf'] for c in confidences),prompts_at_or_above_confidence=sum(c>=policy['conf'] for c in confidences),best_detector_confidence=max(confidences,default=None),filter_details=json.dumps(prompt_details)))
    pd.DataFrame(summaries).to_csv(OUT/'diagnostic_summary.csv',index=False)
    print(f'Diagnosed {number}/{len(targets)}: {key[:8]}')
assert sha(RELEASE/'refinement_policy.json')==policy_hash
(OUT/'protocol.json').write_text(json.dumps(dict(policy_sha256=policy_hash,source_evaluation=str(EVAL),device=DEVICE,scope='Failure diagnosis only. Same fixed thresholds; no labels read, no reference edits, no metric optimization. GPU/CPU replay can differ numerically.'),indent=2))
archive=OUT.with_suffix('.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUT.iterdir():z.write(path,path.name)
display(pd.DataFrame(summaries).drop(columns='filter_details'))
print('Upload this diagnostic ZIP:',archive)
from google.colab import files
files.download(str(archive))
